# Delimiter-guided non-binary in-order oracle with Stanza trees

This notebook mirrors the bottom-up and top-down notebooks, but implements the in-order instantiation of the unified constituency transition system.

The pipeline:

1. read a PTB-style tree with `read_trees`,
2. extract the sentence,
3. derive the static in-order oracle,
4. apply the oracle to reconstruct a Stanza `Tree`,
5. check exact reconstruction against the original tree.

The in-order transition inventory used here is:

\[
\textsc{Shift},\ \textsc{d-Shift},\ \textsc{Nt}(X),\ \textsc{u-Reduce},\ \textsc{Reduce},\ \textsc{d-Reduce},\ \textsc{Finish}.
\]

In-order parsing first constructs the left corner of a constituent, then introduces the mother label, then constructs the remaining children.  Internally, phrase-level reduction applies to the stack pattern:

```text
D | left_child | NT(X) | remaining_child_1 | ... | remaining_child_k
```

and creates:

```text
X(left_child, remaining_child_1, ..., remaining_child_k)
```

`D-REDUCE` preserves the delimiter when the constructed subtree is still the left boundary of a larger pending reduction domain.

All example trees in this notebook are synthetic.

In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from typing import Any, List, Optional, Sequence, Tuple, Union

from stanza.models.constituency.parse_tree import Tree
from stanza.models.constituency.tree_reader import read_trees

OracleOp = Tuple[str, Optional[str]]


## Tree utilities

The helper functions are intentionally the same as in the bottom-up and top-down notebooks, so that the three notebooks can be compared directly.

In [ ]:
def parse_ptb(ptb_string: str) -> Tree:
    """Parse one PTB-style bracketed tree into a Stanza constituency Tree."""
    trees = list(read_trees(ptb_string, use_tqdm=False))
    if not trees:
        raise ValueError(f"No tree found in string: {ptb_string!r}")
    if len(trees) != 1:
        raise ValueError(f"Expected exactly one tree, found {len(trees)}.")
    return trees[0]


def label_of(tree: Tree) -> str:
    """Return the label stored on a Stanza Tree node."""
    return tree.label


def children_of(tree: Tree) -> List[Tree]:
    """Return the children of a Stanza Tree node as a list."""
    return list(tree.children or [])


def make_tree(label: str, children: Optional[Sequence[Tree]] = None) -> Tree:
    """Construct a Stanza Tree node."""
    children = list(children or [])
    try:
        return Tree(label, children)
    except TypeError:
        return Tree(label=label, children=children)


def is_leaf(tree: Tree) -> bool:
    return len(children_of(tree)) == 0


def is_preterminal(tree: Tree) -> bool:
    children = children_of(tree)
    return len(children) == 1 and is_leaf(children[0])


def terminal_word(preterminal: Tree) -> str:
    """Return the terminal word dominated by a preterminal."""
    if not is_preterminal(preterminal):
        raise ValueError(f"Expected a preterminal, found: {tree_to_ptb(preterminal)}")
    return label_of(children_of(preterminal)[0])


def get_sentence(tree: Tree) -> List[str]:
    """Return the terminal yield of a Stanza Tree."""
    if is_leaf(tree):
        return [label_of(tree)]

    words: List[str] = []
    for child in children_of(tree):
        words.extend(get_sentence(child))
    return words


def tree_to_ptb(tree: Tree) -> str:
    """Return a single-line PTB string from a Stanza Tree."""
    if is_leaf(tree):
        return label_of(tree)
    return f"({label_of(tree)} {' '.join(tree_to_ptb(child) for child in children_of(tree))})"


def same_tree(left: Tree, right: Tree) -> bool:
    """Exact structural comparison through the canonical PTB string form."""
    return tree_to_ptb(left) == tree_to_ptb(right)


## Static oracle

The in-order oracle follows a left-corner traversal.

For a preterminal, the oracle emits:

```python
SHIFT(word) or D-SHIFT(word)
NT(POS)
U-REDUCE
```

For a phrasal node, the oracle first visits the leftmost child, then emits `NT(label)`, then visits the remaining children from left to right, and finally emits either `REDUCE` or `D-REDUCE`.

The delimiter policy is the same left-boundary policy used in the bottom-up notebook: a delimiter is inserted before a terminal when that terminal begins a phrasal reduction domain whose boundary may need to survive until a higher node is reduced.

In [ ]:
@dataclass(frozen=True)
class DBoundary:
    """A delimiter introduced by D-SHIFT and consumed by its target node."""

    target_id: int


class TreeNavigator:
    """Small helper for parent and child-index lookup in a Stanza Tree."""

    def __init__(self, root: Tree):
        self.root = root
        self.parent_of: dict[int, Tree] = {}
        self.child_index_of: dict[int, int] = {}
        self._register(root)

    def _register(self, node: Tree) -> None:
        for index, child in enumerate(children_of(node)):
            self.parent_of[id(child)] = node
            self.child_index_of[id(child)] = index
            self._register(child)

    def highest_left_edge_phrasal_ancestor(self, preterminal: Tree) -> Optional[Tree]:
        """
        Return the highest phrasal ancestor reached by moving upward from a
        preterminal while the current node is the first child of its parent.

        The preterminal itself is not a target, because it is built by U-REDUCE.
        """
        current: Tree = preterminal
        target: Optional[Tree] = None

        while id(current) in self.parent_of:
            parent = self.parent_of[id(current)]
            child_index = self.child_index_of[id(current)]

            if child_index != 0:
                break

            if not is_preterminal(parent):
                target = parent

            current = parent

        return target


def get_oracle_inorder(tree: Tree) -> List[OracleOp]:
    """Return the delimiter-guided non-binary in-order oracle for a gold tree.

    The oracle is deterministic for a fixed tree.  It does not encode arity in
    phrase-level reduction actions.  The final action is always FINISH.
    """
    oracle: List[OracleOp] = []
    navigator = TreeNavigator(tree)

    previous_parent_id: Optional[int] = None
    terminal_index = 0

    def shift_word(word: str, parent: Optional[Tree], preterminal: Optional[Tree]) -> Optional[DBoundary]:
        nonlocal previous_parent_id, terminal_index

        current_parent_id = id(parent) if parent is not None else None
        is_first_terminal = terminal_index == 0
        parent_changed = previous_parent_id is not None and current_parent_id != previous_parent_id

        target = None
        if preterminal is not None:
            target = navigator.highest_left_edge_phrasal_ancestor(preterminal)

        use_d_shift = target is not None and (is_first_terminal or parent_changed)

        if use_d_shift:
            oracle.append(("D-SHIFT", word))
            boundary = DBoundary(target_id=id(target))
        else:
            oracle.append(("SHIFT", word))
            boundary = None

        previous_parent_id = current_parent_id
        terminal_index += 1
        return boundary

    def visit(node: Tree, parent: Optional[Tree] = None) -> Optional[DBoundary]:
        if is_leaf(node):
            return shift_word(label_of(node), parent, preterminal=None)

        if is_preterminal(node):
            boundary = shift_word(terminal_word(node), parent, preterminal=node)
            oracle.append(("NT", label_of(node)))
            oracle.append(("U-REDUCE", None))
            return boundary

        children = children_of(node)
        if not children:
            raise ValueError(f"Unexpected empty phrasal node: {label_of(node)!r}")

        active_boundaries: List[DBoundary] = []

        first_boundary = visit(children[0], parent=node)
        if first_boundary is not None:
            active_boundaries.append(first_boundary)

        oracle.append(("NT", label_of(node)))

        for child in children[1:]:
            boundary = visit(child, parent=node)
            if boundary is not None:
                active_boundaries.append(boundary)

        if not active_boundaries:
            oracle.append(("REDUCE", None))
            return None

        if len(active_boundaries) > 1:
            raise ValueError(
                f"Multiple active delimiters reached node {label_of(node)}: {active_boundaries}"
            )

        active = active_boundaries[0]
        if active.target_id == id(node):
            oracle.append(("REDUCE", None))
            return None

        oracle.append(("D-REDUCE", None))
        return active

    final_boundary = visit(tree)
    if final_boundary is not None:
        raise ValueError(f"Unconsumed delimiter boundary: {final_boundary}")

    oracle.append(("FINISH", None))
    return oracle


## Oracle application

The applicator reconstructs a Stanza `Tree` from the sentence and oracle.  Internally, the stack contains completed `Tree` objects and label markers.  Dummy markers represent delimiter boundaries and are never included in the final tree.

For in-order parsing, phrase-level `REDUCE` applies to:

```text
D | left_child | NT(X) | right_child_1 | ... | right_child_k
```

and creates:

```text
X(left_child, right_child_1, ..., right_child_k)
```

For preterminals, `U-REDUCE` applies to:

```text
word | NT(POS)
```

and creates:

```text
POS(word)
```

If the word was introduced by `D-SHIFT`, the delimiter remains below the new preterminal.

In [ ]:
@dataclass(frozen=True)
class LabelMarker:
    label: str
    dummy: bool = False

    def __repr__(self) -> str:
        return "NT()" if self.dummy else f"NT({self.label})"


StackItem = Union[Tree, LabelMarker]


def _is_dummy(item: StackItem) -> bool:
    return isinstance(item, LabelMarker) and item.dummy


def _is_real_marker(item: StackItem) -> bool:
    return isinstance(item, LabelMarker) and not item.dummy


def _item_to_string(item: StackItem) -> str:
    if isinstance(item, LabelMarker):
        return repr(item)
    return tree_to_ptb(item)


def apply_oracle_inorder(
    sentence: Sequence[str],
    oracle: Sequence[OracleOp],
    *,
    verbose: bool = False,
) -> Tree:
    """Apply a delimiter-guided in-order oracle and return the reconstructed Stanza Tree."""

    stack: List[StackItem] = []
    word_index = 0
    finished = False
    final_tree: Optional[Tree] = None

    def stack_to_string() -> str:
        return "[ " + " | ".join(_item_to_string(item) for item in stack) + " ]" if stack else "[]"

    def show(step: int, action: str, value: Optional[str], when: str) -> None:
        if not verbose:
            return
        op = action if value is None else f"{action}({value})"
        remaining = " ".join(sentence[word_index:])
        print(f"{step:03d} {when:<6} {op:<18} stack={stack_to_string()} buffer=[ {remaining} ]")

    def shift(action_value: Optional[str], *, with_delimiter: bool) -> None:
        nonlocal word_index

        if word_index >= len(sentence):
            raise ValueError("SHIFT exceeds sentence length.")

        word = sentence[word_index]
        if action_value is not None and action_value != word:
            raise ValueError(f"SHIFT mismatch: oracle has {action_value!r}, sentence has {word!r}.")

        if with_delimiter:
            stack.append(LabelMarker("", dummy=True))
        stack.append(make_tree(word, []))
        word_index += 1

    def unary_reduce() -> None:
        # Expected in-order preterminal pattern:
        #   ... | word | NT(P)
        if len(stack) < 2 or not _is_real_marker(stack[-1]) or not isinstance(stack[-2], Tree):
            raise ValueError(f"Invalid U-REDUCE configuration: {stack_to_string()}")

        marker = stack.pop()
        child = stack.pop()
        assert isinstance(marker, LabelMarker)
        assert isinstance(child, Tree)

        if not is_leaf(child):
            raise ValueError(f"U-REDUCE only applies to terminals, found: {tree_to_ptb(child)}")

        stack.append(make_tree(marker.label, [child]))

    def reduce_phrase(*, keep_delimiter: bool) -> None:
        # Expected in-order phrase pattern:
        #   prefix | D | left_child | NT(X) | right_child_1 | ... | right_child_k
        marker_index: Optional[int] = None
        delimiter_index: Optional[int] = None

        # The reducible marker is the rightmost real marker whose nearest delimiter
        # to the left begins a domain containing only completed trees, and whose
        # suffix to the top also contains only completed trees.
        for index in range(len(stack) - 1, -1, -1):
            if not _is_real_marker(stack[index]):
                continue

            nearest_delim: Optional[int] = None
            for left in range(index - 1, -1, -1):
                if _is_dummy(stack[left]):
                    nearest_delim = left
                    break

            if nearest_delim is None:
                continue

            left_children = stack[nearest_delim + 1 : index]
            right_children = stack[index + 1 :]

            if left_children and all(isinstance(item, Tree) for item in left_children) and all(
                isinstance(item, Tree) for item in right_children
            ):
                marker_index = index
                delimiter_index = nearest_delim
                break

        if marker_index is None or delimiter_index is None:
            raise ValueError(f"REDUCE cannot find an in-order reduction domain: {stack_to_string()}")

        marker = stack[marker_index]
        assert isinstance(marker, LabelMarker)

        prefix = stack[:delimiter_index]
        children = stack[delimiter_index + 1 : marker_index] + stack[marker_index + 1 :]

        if not children or not all(isinstance(child, Tree) for child in children):
            raise ValueError(f"Invalid children for REDUCE: {stack_to_string()}")

        new_tree = make_tree(marker.label, children)  # type: ignore[arg-type]
        stack[:] = list(prefix)
        if keep_delimiter:
            stack.append(LabelMarker("", dummy=True))
        stack.append(new_tree)

    def finish() -> Tree:
        if word_index != len(sentence):
            raise ValueError(f"FINISH before consuming the sentence: {word_index}/{len(sentence)}")

        if len(stack) != 1 or not isinstance(stack[0], Tree):
            raise ValueError(f"FINISH requires exactly one completed tree: {stack_to_string()}")

        return stack[0]

    for step, (action, value) in enumerate(oracle, start=1):
        if finished:
            raise ValueError(f"Action after FINISH at step {step}: {action}")

        show(step, action, value, "before")

        if action == "SHIFT":
            shift(value, with_delimiter=False)
        elif action == "D-SHIFT":
            shift(value, with_delimiter=True)
        elif action == "NT":
            if value is None:
                raise ValueError("NT requires a label.")
            stack.append(LabelMarker(value))
        elif action == "U-REDUCE":
            unary_reduce()
        elif action == "REDUCE":
            reduce_phrase(keep_delimiter=False)
        elif action == "D-REDUCE":
            reduce_phrase(keep_delimiter=True)
        elif action == "FINISH":
            final_tree = finish()
            finished = True
        else:
            raise ValueError(f"Unknown action: {action}")

        show(step, action, value, "after")

    if not finished or final_tree is None:
        raise ValueError("Oracle ended without FINISH.")

    return final_tree


## End-to-end helpers

`run_pipeline` is the main function used in the examples. It returns the sentence, oracle, reconstructed tree, and exact-match result.

In [ ]:
def format_oracle(oracle: Sequence[OracleOp]) -> str:
    """Format an oracle as one action per line."""
    lines = []
    for i, (action, value) in enumerate(oracle, start=1):
        op = action if value is None else f"{action}({value})"
        lines.append(f"{i:03d}: {op}")
    return "\n".join(lines)


def action_counts(oracle: Sequence[OracleOp]) -> dict[str, int]:
    counts: dict[str, int] = {}
    for action, _ in oracle:
        counts[action] = counts.get(action, 0) + 1
    return counts


def run_pipeline(ptb_string: str, *, verbose: bool = False) -> dict[str, Any]:
    gold_tree = parse_ptb(ptb_string)
    sentence = get_sentence(gold_tree)
    oracle = get_oracle_inorder(gold_tree)
    reconstructed_tree = apply_oracle_inorder(sentence, oracle, verbose=verbose)

    return {
        "sentence": sentence,
        "oracle": oracle,
        "gold_tree": gold_tree,
        "reconstructed_tree": reconstructed_tree,
        "exact_match": same_tree(gold_tree, reconstructed_tree),
        "action_counts": action_counts(oracle),
    }


## Minimal example

Run the synthetic tree below to inspect the oracle actions and reconstructed tree.

In [ ]:
ptb_string = '(TOP (S (NP (NNP Mira) (NNP Vale)) (VP (VBZ builds) (NP (NN robots))) (. .)))'

result = run_pipeline(ptb_string)

print("Sentence:", result["sentence"])
print("Action counts:", result["action_counts"])
print("Exact reconstruction:", result["exact_match"])
print()
print(format_oracle(result["oracle"]))
print()
print("Reconstructed tree:")
print(tree_to_ptb(result["reconstructed_tree"]))

assert result["exact_match"]


## Regression examples

These synthetic trees exercise unary chains, branching, and punctuation.

In [ ]:
examples = [
    '(TOP (S (NP (NNP Mira) (NNP Vale)) (VP (VBZ builds) (NP (NN robots))) (. .)))',
    '(TOP (S (NP (DT The) (JJ young) (NN fox)) (VP (VBD watched) (SBAR (IN as) (S (NP (DT a) (NN bird)) (VP (VBD flew) (PP (IN over) (NP (DT the) (NN pond))))))) (. .)))',
    '(TOP (S (VP (VB Wait))))',
    '(TOP (S (NP (NNP Ivo)) (VP (VBZ packs) (NP (NNS maps)) (NP (NNS snacks)) (PP (IN for) (NP (NN travel)))) (. .)))',
]

for index, example in enumerate(examples, start=1):
    result = run_pipeline(example)
    print(
        f"example {index}: tokens={len(result['sentence'])}, "
        f"actions={len(result['oracle'])}, exact_match={result['exact_match']}, "
        f"counts={result['action_counts']}"
    )
    assert result["exact_match"]


## Optional verbose trace

Set `verbose=True` to inspect the stack and buffer before and after every transition.

In [ ]:
# _ = run_pipeline(ptb_string, verbose=True)
